# Data Prep v4 — 端到端数据准备管线

**方案四（新版）**全流程整合：
1. 语料 & 查询集构建（Vul + BFP + white/gray/black query_set）
2. 三个向量库构建（BFP-APR / Vul-Vinj / 历史查询）
3. 关联 CVE 实例与历史查询（CVE↔历史查询匹配 + unsafe_query_set_v4）
4. 代理查询集 & 测试集划分（unsafe 按 8:2 切分）
5. 投毒目标选择与正样本构建（反向索引 + τ 过滤 + 聚类 + 预算加权采样）
6. 相关漏洞检索（Vinj 检索 + 加权采样 relevant_vul）

> **使用方法**：按顺序运行所有 cell；每个写盘/建库步骤均有幂等检查，已完成则自动跳过。
> 盒设置（white/gray/black）由 `configs/database/bfp/query.yml` 的 `data.corpus` 路径自动解析。

## Section 0 — 全局初始化

In [1]:
# ========== Section 0: 全局初始化 ==========
import os
import sys
import gc
import json
import random
import torch
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans

# ── 项目根目录上溯 ──
PROJECT_ROOT = Path(os.path.abspath("")).resolve()
while not (PROJECT_ROOT / "configs").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"PROJECT_ROOT: {PROJECT_ROOT}")

from src.utils.io import load_json, load_jsonl, load_json_stream, save_json, load_yaml
from src.utils.log import setup_logging
from src.models.retriever.Base import create_embedder
from src.rag.bfp.milvus_client import BFPMilvusClient
from src.rag.bfp.retriever import BFPRetriever
from src.rag.vul.milvus_client import VulMilvusClient
from src.rag.vul.retriever import VulRetriever

setup_logging()


# ── 嵌入模型按需加载/释放的辅助函数 ──
def load_embedder_from_db_config(db_config: dict):
    """从数据库配置的 database.model 字段加载嵌入模型。"""
    os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
    cfg_path = PROJECT_ROOT / db_config["database"]["model"]
    return create_embedder(load_yaml(str(cfg_path)))

def release_embedder(emb) -> None:
    """释放嵌入模型并清空 GPU 显存。"""
    del emb
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("[GPU] 嵌入模型已释放")


# ── 从 query.yml 的 corpus 路径解析盒设置（white/gray/black）──
def derive_box_from_path(corpus_path: str) -> str:
    """从形如 data/query/{box}/query_set.jsonl 的路径解析盒名（white/gray/black）。

    作为 Section 3~6 全流程输出路径的单一真源，避免盒设置在多处硬编码不一致。
    """
    parts = Path(str(corpus_path).replace("\\", "/")).parts
    if "query" in parts:
        idx = parts.index("query")
        if idx + 1 < len(parts) and parts[idx + 1] in ("white", "gray", "black"):
            return parts[idx + 1]
    raise ValueError(f"无法从路径解析盒设置(white/gray/black): {corpus_path}")

PROJECT_ROOT: <PROJECT_ROOT>


## Section 1 — 语料 & 查询集构建

### Section 1.1 — Vul 语料构建

In [2]:
# ========== Section 1.1: Vul 语料构建 ==========
# 产出：data/corpus/vul/clustered_rag_data.jsonl

VUL_CORPUS_OUT = PROJECT_ROOT / "data/corpus/vul/clustered_rag_data.jsonl"

if VUL_CORPUS_OUT.exists():
    print(f"[SKIP] Vul 语料已存在：{VUL_CORPUS_OUT}")
    df_vul = pd.read_json(str(VUL_CORPUS_OUT), lines=True)
    print(f"       加载 {len(df_vul)} 条")
else:
    # ── 1. 加载原始数据集（已由 vul/dataset_formatting.ipynb 预先格式化）──
    bigvul_df   = pd.read_json(str(PROJECT_ROOT / "data/raw/vul/BigVul.jsonl"),   lines=True)
    primevul_df = pd.read_json(str(PROJECT_ROOT / "data/raw/vul/PrimeVul.jsonl"), lines=True)
    df_vul = pd.concat([bigvul_df, primevul_df], ignore_index=True)
    print(f"原始合并: {len(df_vul)} 条")

    # ── 2. 按 vul_code + cwe_id 去重（同一漏洞代码可能对应不同 CWE，需保留）──
    before = len(df_vul)
    df_vul = df_vul.drop_duplicates(subset=["vul_code", "cwe_id"], keep="first").reset_index(drop=True)
    print(f"去重: {before} → {len(df_vul)} (移除 {before - len(df_vul)})")

    # ── 3. 嵌入 + KMeans 聚类 ──
    all_codes = df_vul["vul_code"].astype(str).tolist()
    vul_embedder = load_embedder_from_db_config(
        {"database": {"model": "configs/models/harrier-oss-v1-0.6b.yml"}}
    )
    encoded = np.asarray(vul_embedder.embed_documents(all_codes))
    release_embedder(vul_embedder)  # KMeans 不需要 GPU，提前释放显存
    print(f"嵌入维度: {encoded.shape}")

    kmeans = KMeans(n_clusters=5, random_state=42)
    kmeans.fit(encoded)
    df_vul["cluster_id"] = kmeans.labels_
    print(f"KMeans 聚类完成，各簇分布:\n{df_vul['cluster_id'].value_counts().sort_index()}")

    # ── 4. 写出 ──
    VUL_CORPUS_OUT.parent.mkdir(parents=True, exist_ok=True)
    df_vul.to_json(str(VUL_CORPUS_OUT), orient="records", lines=True)
    print(f"✅ 已写入 {len(df_vul)} 条 → {VUL_CORPUS_OUT}")

df_vul.head()

[SKIP] Vul 语料已存在：<PROJECT_ROOT>/data/corpus/vul/clustered_rag_data.jsonl
       加载 17576 条


,id,source,language,cwe_id,cve_id,cwe_desc,cve_desc,vul_code,fixed_code,diff,cluster_id
0,0,BigVul,C,CWE-264,CVE-2015-8467,"- CWE ID: CWE-264\n- Name: Permissions, Privil...",The samldb_check_user_account_control_acl func...,static int samldb_check_user_account_control_a...,static int samldb_check_user_account_control_a...,"--- vul_code\n+++ fixed_code\n@@ -9,12 +9,15 @...",2
1,1,BigVul,C,CWE-119,CVE-2015-8382,- CWE ID: CWE-119\n- Name: Improper Restrictio...,The match function in pcre_exec.c in PCRE befo...,PHPAPI void php_pcre_match_impl(pcre_cache_ent...,PHPAPI void php_pcre_match_impl(pcre_cache_ent...,"--- vul_code\n+++ fixed_code\n@@ -87,6 +87,7 @...",3
2,2,BigVul,C,CWE-119,CVE-2013-6712,- CWE ID: CWE-119\n- Name: Improper Restrictio...,The scan function in ext/date/lib/parse_iso_in...,static int scan(Scanner *s) {\n uchar *cursor...,static int scan(Scanner *s) {\n uchar *cursor...,"--- vul_code\n+++ fixed_code\n@@ -147,7 +147,7...",3
3,3,BigVul,C,CWE-310,CVE-2013-6449,- CWE ID: CWE-310\n- Name: Cryptographic Issue...,The ssl_get_algorithm2 function in ssl/s3_lib....,long ssl_get_algorithm2(SSL *s) {\n long alg2...,long ssl_get_algorithm2(SSL *s) {\n long alg2...,"--- vul_code\n+++ fixed_code\n@@ -1,6 +1,6 @@\...",4
4,4,BigVul,C,CWE-119,CVE-2013-6420,- CWE ID: CWE-119\n- Name: Improper Restrictio...,The asn1_time_to_time_t function in ext/openss...,static time_t asn1_time_to_time_t(ASN1_UTCTIME...,static time_t asn1_time_to_time_t(ASN1_UTCTIME...,"--- vul_code\n+++ fixed_code\n@@ -13,20 +13,30...",3


### Section 1.2 — BFP 语料构建

In [3]:
# ========== Section 1.2: BFP 语料构建 ==========
# 产出：data/corpus/bfp/rag_data.jsonl

CORPUS_SIZE = 50000
SEED = 42
BFP_CORPUS_OUT = PROJECT_ROOT / "data/corpus/bfp/rag_data.jsonl"

# ── 工具函数 ──
def extract_fields(record: dict, source: str) -> dict:
    return {
        "language": record["language"],
        "buggy_code": record["buggy_code"],
        "fixed_code": record["fixed_code"],
        "source": source,
    }

def assign_ids(data: list) -> list:
    return [{"id": i} | item for i, item in enumerate(data, 1)]

def write_jsonl(path, data: list):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")
    print(f"  ✅ 已写入 {len(data)} 条 → {path}")

if BFP_CORPUS_OUT.exists():
    print(f"[SKIP] BFP 语料已存在：{BFP_CORPUS_OUT}")
    corpus_dataset = load_jsonl(str(BFP_CORPUS_OUT))
    print(f"       加载 {len(corpus_dataset)} 条")
else:
    random.seed(SEED)

    # ── 读取原始数据（已由 bfp/dataset_formatting.ipynb 预先格式化）──
    coconut_data = []
    with open(PROJECT_ROOT / "data/raw/bfp/CoCoNut.jsonl", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                coconut_data.append(json.loads(line))
    with open(PROJECT_ROOT / "data/raw/bfp/Codeflaws.json", encoding="utf-8") as f:
        codeflaws_data = json.load(f)
    with open(PROJECT_ROOT / "data/raw/bfp/Deepfix.json", encoding="utf-8") as f:
        deepfix_data = json.load(f)

    random.shuffle(coconut_data)
    random.shuffle(codeflaws_data)
    random.shuffle(deepfix_data)

    print(f"原始数据: CoCoNut={len(coconut_data)}, Codeflaws={len(codeflaws_data)}, DeepFix={len(deepfix_data)}")

    # ── 按占比分配语料库样本数 ──
    TOTAL_RAW = len(coconut_data) + len(codeflaws_data) + len(deepfix_data)
    CORPUS_COCONUT   = round(CORPUS_SIZE * len(coconut_data)   / TOTAL_RAW)
    CORPUS_CODEFLAWS = round(CORPUS_SIZE * len(codeflaws_data) / TOTAL_RAW)
    CORPUS_DEEPFIX   = CORPUS_SIZE - CORPUS_COCONUT - CORPUS_CODEFLAWS

    print(f"语料库分配: CoCoNut={CORPUS_COCONUT}, Codeflaws={CORPUS_CODEFLAWS}, DeepFix={CORPUS_DEEPFIX}")

    corpus_coconut   = coconut_data[:CORPUS_COCONUT]
    corpus_codeflaws = codeflaws_data[:CORPUS_CODEFLAWS]
    corpus_deepfix   = deepfix_data[:CORPUS_DEEPFIX]

    corpus_dataset = assign_ids(
        [extract_fields(r, "CoCoNut")   for r in corpus_coconut] +
        [extract_fields(r, "Codeflaws") for r in corpus_codeflaws] +
        [extract_fields(r, "DeepFix")   for r in corpus_deepfix]
    )
    assert len(corpus_dataset) == CORPUS_SIZE

    write_jsonl(BFP_CORPUS_OUT, corpus_dataset)

print(f"BFP 语料库规模: {len(corpus_dataset)} 条")

[SKIP] BFP 语料已存在：<PROJECT_ROOT>/data/corpus/bfp/rag_data.jsonl


2026-07-25 11:36:46 [INFO] src.utils.io: Loaded 50000 records from <PROJECT_ROOT>/data/corpus/bfp/rag_data.jsonl


       加载 50000 条
BFP 语料库规模: 50000 条


### Section 1.3 — BFP 查询集构建（white / gray / black）

In [4]:
# ========== Section 1.3: BFP 查询集构建 ==========
# 产出：
#   data/query/white/query_set.jsonl  （完全 ⊆ 语料库）
#   data/query/gray/query_set.jsonl   （OVERLAP_RATE 来自语料库 + 余下来自语料库外）
#   data/query/black/query_set.jsonl  （全部来自语料库外）

QUERY_SIZE    = 50000
OVERLAP_RATE  = 0.1
SEED          = 42

WHITE_QUERY_OUT = PROJECT_ROOT / "data/query/white/query_set.jsonl"
GRAY_QUERY_OUT  = PROJECT_ROOT / "data/query/gray/query_set.jsonl"
BLACK_QUERY_OUT = PROJECT_ROOT / "data/query/black/query_set.jsonl"

all_done = WHITE_QUERY_OUT.exists() and GRAY_QUERY_OUT.exists() and BLACK_QUERY_OUT.exists()

if all_done:
    print("[SKIP] 三套查询集已存在")
    white_dataset = load_jsonl(str(WHITE_QUERY_OUT))
    gray_dataset  = load_jsonl(str(GRAY_QUERY_OUT))
    black_dataset = load_jsonl(str(BLACK_QUERY_OUT))
    print(f"       white={len(white_dataset)}, gray={len(gray_dataset)}, black={len(black_dataset)}")
else:
    rng = random.Random(SEED)

    # ── corpus_pool：直接使用已生成的语料库 ──
    corpus_pool = corpus_dataset[:]  # 来自 Section 1.2，已格式化

    # ── outside_pool：原始数据集 − 语料库 ──
    # 用 buggy_code 做精确去重（语料库内条目均来自原始数据集前 N 条）
    corpus_buggy_set = {item["buggy_code"] for item in corpus_pool}

    # 重新读取原始数据（如果全局变量还在则直接用）
    def _load_raw_bfp():
        coc = []
        with open(PROJECT_ROOT / "data/raw/bfp/CoCoNut.jsonl", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    coc.append(json.loads(line))
        with open(PROJECT_ROOT / "data/raw/bfp/Codeflaws.json", encoding="utf-8") as f:
            clf = json.load(f)
        with open(PROJECT_ROOT / "data/raw/bfp/Deepfix.json", encoding="utf-8") as f:
            dpf = json.load(f)
        return coc, clf, dpf

    def infer_source(record: dict) -> str:
        if "user_id" in record:
            return "DeepFix"
        if "bug_type" in record:
            return "Codeflaws"
        return "CoCoNut"

    try:
        # 如果 Section 1.2 的变量还在内存中直接利用
        _raw_all = (
            [extract_fields(r, "CoCoNut")   for r in coconut_data] +
            [extract_fields(r, "Codeflaws") for r in codeflaws_data] +
            [extract_fields(r, "DeepFix")   for r in deepfix_data]
        )
    except NameError:
        coc, clf, dpf = _load_raw_bfp()
        _raw_all = (
            [extract_fields(r, "CoCoNut")   for r in coc] +
            [extract_fields(r, "Codeflaws") for r in clf] +
            [extract_fields(r, "DeepFix")   for r in dpf]
        )

    outside_pool = [
        item for item in _raw_all
        if item["buggy_code"] not in corpus_buggy_set
    ]
    print(f"outside_pool 大小: {len(outside_pool)} 条（已由 bfp/dataset_formatting.ipynb 预先格式化）")

    # ── 抽取三套查询集 ──
    overlap_n = int(QUERY_SIZE * OVERLAP_RATE)   # gray 中来自语料库的部分
    outside_n = QUERY_SIZE - overlap_n           # gray 中来自语料库外的部分

    assert len(corpus_pool) >= QUERY_SIZE, "corpus_pool 不足以支撑 white 查询集"
    assert len(outside_pool) >= QUERY_SIZE + overlap_n, "outside_pool 数量不足"

    white_raw  = rng.sample(corpus_pool, QUERY_SIZE)
    gray_corp  = rng.sample(corpus_pool, overlap_n)
    gray_out   = rng.sample(outside_pool, outside_n)
    gray_raw   = gray_corp + gray_out
    black_raw  = rng.sample(outside_pool, QUERY_SIZE)

    white_dataset = assign_ids(white_raw)
    gray_dataset  = assign_ids(gray_raw)
    black_dataset = assign_ids(black_raw)

    write_jsonl(WHITE_QUERY_OUT, white_dataset)
    write_jsonl(GRAY_QUERY_OUT,  gray_dataset)
    write_jsonl(BLACK_QUERY_OUT, black_dataset)

print(f"\n查询集规模汇总:")
print(f"  white : {len(white_dataset)} 条 → {WHITE_QUERY_OUT}")
print(f"  gray  : {len(gray_dataset)} 条 → {GRAY_QUERY_OUT}")
print(f"  black : {len(black_dataset)} 条 → {BLACK_QUERY_OUT}")

[SKIP] 三套查询集已存在


2026-07-25 11:37:30 [INFO] src.utils.io: Loaded 50000 records from <PROJECT_ROOT>/data/query/white/query_set.jsonl
2026-07-25 11:37:49 [INFO] src.utils.io: Loaded 50000 records from <PROJECT_ROOT>/data/query/gray/query_set.jsonl
2026-07-25 11:38:12 [INFO] src.utils.io: Loaded 50000 records from <PROJECT_ROOT>/data/query/black/query_set.jsonl


       white=50000, gray=50000, black=50000

查询集规模汇总:
  white : 50000 条 → <PROJECT_ROOT>/data/query/white/query_set.jsonl
  gray  : 50000 条 → <PROJECT_ROOT>/data/query/gray/query_set.jsonl
  black : 50000 条 → <PROJECT_ROOT>/data/query/black/query_set.jsonl


## Section 2 — 向量库构建

### Section 2.1 — BFP-APR 向量库

配置文件：`configs/database/bfp/apr.yml`  
语料库和模型文件需要在配置文件中进行设置

In [5]:
# ========== Section 2.1: 构建 BFP-APR 向量库 ==========
# 模型和 URI 均从 apr.yml 的 database.model / database.uri 读取，无需手动覆写

apr_db_config = load_yaml(str(PROJECT_ROOT / "configs/database/bfp/apr.yml"))

# 路径绝对化
if not os.path.isabs(apr_db_config["database"]["uri"]):
    apr_db_config["database"]["uri"] = str(PROJECT_ROOT / apr_db_config["database"]["uri"])
if not os.path.isabs(apr_db_config["data"]["corpus"]):
    apr_db_config["data"]["corpus"] = str(PROJECT_ROOT / apr_db_config["data"]["corpus"])

apr_milvus = BFPMilvusClient(apr_db_config)
col_name = apr_db_config["database"]["collection_name"]

already_built = (
    apr_milvus.client.has_collection(col_name)
    and apr_milvus.get_collection_stats()["row_count"] > 0
)

if already_built:
    rows = apr_milvus.get_collection_stats()["row_count"]
    print(f"[SKIP] BFP-APR 向量库已存在，共 {rows} 条")
else:
    kb_data = load_jsonl(apr_db_config["data"]["corpus"])
    print(f"构建 BFP-APR 向量库，数据量: {len(kb_data)} 条...")
    local_embedder = load_embedder_from_db_config(apr_db_config)
    apr_milvus.build_knowledge_base(local_embedder, kb_data)
    
    release_embedder(local_embedder)
    print(f"✅ BFP-APR 向量库构建完成")

apr_milvus.close()
print("BFP-APR 向量库 Done")

2026-07-25 11:38:12 [INFO] src.rag.bfp.milvus_client: 从模型配置 <PROJECT_ROOT>/configs/models/harrier-oss-v1-0.6b.yml 读取 dim=1024
2026-07-25 11:38:12 [INFO] src.rag.bfp.milvus_client: Connecting to Milvus at <PROJECT_ROOT>/data/milvus/bfp/apr/harrier-oss-v1-0.6b.db
2026-07-25 11:38:12 [INFO] src.rag.bfp.milvus_client: Milvus client connected
2026-07-25 11:38:32 [INFO] src.rag.bfp.milvus_client: Milvus client closed


[SKIP] BFP-APR 向量库已存在，共 50000 条
BFP-APR 向量库 Done


### Section 2.2 — Vul-Vinj 向量库

配置文件：`configs/database/vul/vinj.yml`  
语料库和模型文件需要在配置文件中进行设置

In [6]:
# ========== Section 2.2: 构建 Vul-Vinj 向量库 ==========

vinj_db_config = load_yaml(str(PROJECT_ROOT / "configs/database/vul/vinj.yml"))

# 路径绝对化
if not os.path.isabs(vinj_db_config["database"]["uri"]):
    vinj_db_config["database"]["uri"] = str(PROJECT_ROOT / vinj_db_config["database"]["uri"])
if not os.path.isabs(vinj_db_config["data"]["corpus"]):
    vinj_db_config["data"]["corpus"] = str(PROJECT_ROOT / vinj_db_config["data"]["corpus"])

vinj_milvus = VulMilvusClient(vinj_db_config)
col_name = vinj_db_config["database"]["collection_name"]

already_built = (
    vinj_milvus.client.has_collection(col_name)
    and vinj_milvus.get_collection_stats()["row_count"] > 0
)

if already_built:
    rows = vinj_milvus.get_collection_stats()["row_count"]
    print(f"[SKIP] Vul-Vinj 向量库已存在，共 {rows} 条")
else:
    kb_data = load_jsonl(vinj_db_config["data"]["corpus"])
    print(f"构建 Vul-Vinj 向量库，数据量: {len(kb_data)} 条...")
    local_embedder = load_embedder_from_db_config(vinj_db_config)
    vinj_milvus.build_knowledge_base(local_embedder, kb_data)
    release_embedder(local_embedder)
    print(f"✅ Vul-Vinj 向量库构建完成")

vinj_milvus.close()
print("Vul-Vinj 向量库 Done")

2026-07-25 11:38:49 [INFO] src.rag.vul.milvus_client: 从模型配置 <PROJECT_ROOT>/configs/models/harrier-oss-v1-0.6b.yml 读取 dim=1024
2026-07-25 11:38:49 [INFO] src.rag.vul.milvus_client: Connecting to Milvus at <PROJECT_ROOT>/data/milvus/vul/harrier-oss-v1-0.6b.db
2026-07-25 11:38:49 [INFO] src.rag.vul.milvus_client: Milvus client connected
2026-07-25 11:38:58 [INFO] src.rag.vul.milvus_client: Milvus client closed


[SKIP] Vul-Vinj 向量库已存在，共 17576 条
Vul-Vinj 向量库 Done


### Section 2.3 — 历史查询向量库

配置文件：`configs/database/bfp/query.yml`  
语料库和模型文件需要在配置文件中进行设置

In [7]:
# ========== Section 2.3: 构建历史查询向量库 ==========
query_db_config = load_yaml(str(PROJECT_ROOT / "configs/database/bfp/query.yml"))

# 路径绝对化
if not os.path.isabs(query_db_config["database"]["uri"]):
    query_db_config["database"]["uri"] = str(PROJECT_ROOT / query_db_config["database"]["uri"])
if not os.path.isabs(query_db_config["data"]["corpus"]):
    query_db_config["data"]["corpus"] = str(PROJECT_ROOT / query_db_config["data"]["corpus"])
if not os.path.isabs(query_db_config["data"]["query"]):
    query_db_config["data"]["query"] = str(PROJECT_ROOT / query_db_config["data"]["query"])

query_milvus = BFPMilvusClient(query_db_config)
col_name = query_db_config["database"]["collection_name"]

already_built = (
    query_milvus.client.has_collection(col_name)
    and query_milvus.get_collection_stats()["row_count"] > 0
)

if already_built:
    rows = query_milvus.get_collection_stats()["row_count"]
    print(f"[SKIP] 历史查询向量库已存在，共 {rows} 条")
else:
    kb_data = load_jsonl(query_db_config["data"]["corpus"])
    print(f"构建历史查询向量库，数据量: {len(kb_data)} 条...")
    local_embedder = load_embedder_from_db_config(query_db_config)
    query_milvus.build_knowledge_base(local_embedder, kb_data)
    release_embedder(local_embedder)
    print(f"✅ 历史查询向量库构建完成")

query_milvus.close()
print("历史查询向量库 Done")

2026-07-25 11:39:37 [INFO] src.rag.bfp.milvus_client: 从模型配置 <PROJECT_ROOT>/configs/models/harrier-oss-v1-0.6b.yml 读取 dim=1024
2026-07-25 11:39:37 [INFO] src.rag.bfp.milvus_client: Connecting to Milvus at <PROJECT_ROOT>/data/milvus/bfp/query/black/harrier-oss-v1-0.6b.db
2026-07-25 11:39:37 [INFO] src.rag.bfp.milvus_client: Milvus client connected
2026-07-25 11:39:52 [INFO] src.rag.bfp.milvus_client: Milvus client closed


[SKIP] 历史查询向量库已存在，共 50000 条
历史查询向量库 Done


## Section 3 — 关联 CVE 实例与历史查询

### Section 3.1 — CVE 实例 ↔ 历史查询匹配

以 Vul 语料库中每条 CVE 实例的 `vul_code` 作为查询，从历史查询向量库（`query.yml`）检索相关 BFP（历史查询），并将结果保存为检索结果文件。盒设置（white/gray/black）在此从 `query.yml` 的 `data.corpus` 路径解析，供后续 Section 3~6 复用。

In [8]:
# ========== Section 3.1: CVE 实例 ↔ 历史查询匹配 ==========
# 使用 query.yml 配置（data.corpus=历史查询集, data.query=CVE实例, query_target=vul_code）

query_db_config_s3 = load_yaml(str(PROJECT_ROOT / "configs/database/bfp/query.yml"))

# ── 从 query.yml 的 corpus 路径解析盒设置（供 Section 3~6 全流程复用）──
BOX = derive_box_from_path(query_db_config_s3["data"]["corpus"])
print(f"盒设置（white/gray/black）: {BOX}")

# 路径绝对化
if not os.path.isabs(query_db_config_s3["database"]["uri"]):
    query_db_config_s3["database"]["uri"] = str(PROJECT_ROOT / query_db_config_s3["database"]["uri"])
if not os.path.isabs(query_db_config_s3["data"]["corpus"]):
    query_db_config_s3["data"]["corpus"] = str(PROJECT_ROOT / query_db_config_s3["data"]["corpus"])
if not os.path.isabs(query_db_config_s3["data"]["query"]):
    query_db_config_s3["data"]["query"] = str(PROJECT_ROOT / query_db_config_s3["data"]["query"])

# 推导输出文件路径
top_k_s3              = query_db_config_s3["retrieval"].get("top_k", 10)
query_target_s3       = query_db_config_s3["retrieval"].get("query_target", "vul_code")
retrieval_strategy_s3 = query_db_config_s3["retrieval"].get("retrieval_strategy", "dense")
alpha_str_s3          = "1" if retrieval_strategy_s3 == "dense" else (
                        "0" if retrieval_strategy_s3 == "sparse" else
                        str(query_db_config_s3["retrieval"].get("alpha_weight", 0.5)))

# 加载查询数据（CVE 实例）
query_data_s3 = load_jsonl(query_db_config_s3["data"]["query"])
print(f"CVE 实例查询集: {len(query_data_s3)} 条")

# 加载 embedder 以获取模型名（用于构造结果文件名）
query_embedder_s3  = load_embedder_from_db_config(query_db_config_s3)
model_short_name   = query_embedder_s3.get_model_name(with_provider=False)

retrieval_results_filename = (
    f"num_{len(query_data_s3)}_alpha_{alpha_str_s3}"
    f"_target_{query_target_s3}_retriever_{model_short_name}_retrieval_results.json"
)
PROXY_RETRIEVAL_RESULTS_PATH = PROJECT_ROOT / "data/query" / BOX / retrieval_results_filename
print(f"检索结果将保存至: {PROXY_RETRIEVAL_RESULTS_PATH}")

if PROXY_RETRIEVAL_RESULTS_PATH.exists():
    release_embedder(query_embedder_s3)
    print(f"[SKIP] 检索结果文件已存在")
    proxy_retrieval_results = load_json_stream(str(PROXY_RETRIEVAL_RESULTS_PATH))
    print(f"       加载 {len(proxy_retrieval_results)} 条")
else:
    # 打开历史查询向量库并执行检索
    query_milvus_s3 = BFPMilvusClient(query_db_config_s3)
    retriever_s3    = BFPRetriever(query_embedder_s3, query_milvus_s3)

    print(f"执行密集检索: top_k={top_k_s3}, query_target={query_target_s3}")
    proxy_retrieval_results = retriever_s3.dense_retrieve(
        query_data_s3, top_k=top_k_s3, query_target=query_target_s3
    )

    PROXY_RETRIEVAL_RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
    save_json(proxy_retrieval_results, str(PROXY_RETRIEVAL_RESULTS_PATH))
    print(f"✅ 检索结果已保存: {len(proxy_retrieval_results)} 条 → {PROXY_RETRIEVAL_RESULTS_PATH}")

    query_milvus_s3.close()
    release_embedder(query_embedder_s3)

print("Section 3.1 Done")

盒设置（white/gray/black）: black


2026-07-25 11:40:05 [INFO] src.utils.io: Loaded 17576 records from <PROJECT_ROOT>/data/corpus/vul/clustered_rag_data.jsonl


CVE 实例查询集: 17576 条


2026-07-25 11:40:06 [INFO] src.utils.device: nvidia-smi 探测完成: 健康 GPU = [0]
2026-07-25 11:40:06 [WARNING] src.utils.device: 设备池预筛选排除 cuda:1（nvidia-smi 判定为不健康）
2026-07-25 11:40:06 [WARNING] src.utils.device: 设备池预筛选排除 cuda:2（nvidia-smi 判定为不健康）
2026-07-25 11:40:06 [WARNING] src.utils.device: 设备池预筛选排除 cuda:3（nvidia-smi 判定为不健康）
2026-07-25 11:40:06 [WARNING] src.utils.device: 设备池预筛选排除 cuda:4（nvidia-smi 判定为不健康）
2026-07-25 11:40:06 [WARNING] src.utils.device: 设备池预筛选排除 cuda:5（nvidia-smi 判定为不健康）
2026-07-25 11:40:06 [WARNING] src.utils.device: 设备池预筛选排除 cuda:6（nvidia-smi 判定为不健康）
2026-07-25 11:40:06 [WARNING] src.utils.device: 设备池预筛选排除 cuda:7（nvidia-smi 判定为不健康）
2026-07-25 11:40:06 [WARNING] src.utils.device: 设备池预筛选排除 cuda:8（nvidia-smi 判定为不健康）
2026-07-25 11:40:06 [WARNING] src.utils.device: 设备池预筛选排除 cuda:9（nvidia-smi 判定为不健康）
2026-07-25 11:40:07 [INFO] src.utils.device: 设备池探测成功: cuda:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8187 MiB)
<CONDA_ENV>/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning:

检索结果将保存至: <PROJECT_ROOT>/data/query/black/num_17576_alpha_1_target_vul_code_retriever_harrier-oss-v1-0.6b_retrieval_results.json
[GPU] 嵌入模型已释放
[SKIP] 检索结果文件已存在


2026-07-25 11:40:41 [INFO] src.utils.io: Stream-loaded 17576 records from <PROJECT_ROOT>/data/query/black/num_17576_alpha_1_target_vul_code_retriever_harrier-oss-v1-0.6b_retrieval_results.json


       加载 17576 条
Section 3.1 Done


### Section 3.2 — 含安全隐患的历史查询集（unsafe_query_set_v4）

遍历 CVE↔历史查询检索结果，按目标 CWE 汇总命中的 BFP：仅保留以 `score >= RELEVANCE_THRESHOLD`、`rank <= TOP_K_FILTER` 被该 CWE 类型 CVE 实例召回的 BFP；统计每个 BFP 关联到的**不同 CVE 数量**，过滤掉关联数 `< CVE_ASSOCIATION_THRESHOLD` 的 BFP，并按 CWE 分类去重存储为 `unsafe_query_set_v4.json`。

In [9]:
# ========== Section 3.2: 含安全隐患的历史查询集（unsafe_query_set_v4） ==========

# CWE_LIST                  = ["CWE-125", "CWE-20", "CWE-119", "CWE-787", "CWE-416"]
# CWE_LIST = ["CWE-787", "CWE-416", "CWE-125", "CWE-120", "CWE-476", "CWE-122", "CWE-20", "CWE-200"]
# CWE_LIST = ["CWE-787", "CWE-416", "CWE-125", "CWE-476", "CWE-20", "CWE-284", "CWE-200"]
# CWE_LIST = ["CWE-787", "CWE-416", "CWE-125", "CWE-476", "CWE-20", "CWE-200"]
# CWE_LIST = ["CWE-787", "CWE-416", "CWE-125", "CWE-476", "CWE-20", "CWE-200", "CWE-119", "CWE-190", "CWE-362", "CWE-120"]
CWE_LIST = ["CWE-787", "CWE-416", "CWE-125", "CWE-476", "CWE-20", "CWE-200", "CWE-119", "CWE-362"]


RELEVANCE_THRESHOLD       = 0.8
# RELEVANCE_THRESHOLD       = 0.75
TOP_K_FILTER              = 10
CVE_ASSOCIATION_THRESHOLD = 2   # 关联数量阈值 τ：保留关联到的不同 CVE 数 >= τ 的 BFP

UNSAFE_QUERY_SET_PATH = PROJECT_ROOT / "data/query" / BOX / "unsafe_query_set_v4.json"


def collect_unsafe_bfps_for_cwe(
    results: list, cwe_id: str, threshold: float, top_k: int, tau: int
) -> list:
    """从 CVE↔历史查询检索结果中按 CWE 汇总含安全隐患的 BFP。

    对属于 cwe_id 的每个 CVE 实例，统计其在 rank<=top_k 且 score>=threshold 内
    召回的 BFP；记录每个 BFP 关联到的不同 cve_id 集合，并保留其最高分命中。
    最后仅保留关联到的不同 CVE 数 >= tau 的 BFP（按 doc_id 去重）。
    """
    bfp_best = {}                 # doc_id -> 最高分 hit
    bfp_cves = defaultdict(set)   # doc_id -> {cve_id, ...}
    for entry in results:
        ent = entry.get("entity", {})
        if ent.get("cwe_id") != cwe_id:
            continue
        cve_id = (ent.get("cve_id") or "").strip()
        for rank, hit in enumerate(entry.get("retrieval_results", []), start=1):
            if rank > top_k:
                break
            if hit["score"] < threshold:
                continue
            doc_id = hit["doc_id"]
            if cve_id:
                bfp_cves[doc_id].add(cve_id)
            if doc_id not in bfp_best or hit["score"] > bfp_best[doc_id]["score"]:
                bfp_best[doc_id] = hit

    return [
        {"id": h["doc_id"], "language": h.get("language", "C"), "source": h["source"],
         "buggy_code": h["buggy_code"], "fixed_code": h["fixed_code"]}
        for doc_id, h in bfp_best.items()
        if len(bfp_cves[doc_id]) >= tau
    ]


if UNSAFE_QUERY_SET_PATH.exists():
    print(f"[SKIP] 含安全隐患的历史查询集已存在：{UNSAFE_QUERY_SET_PATH}")
    unsafe_query_set = load_json(str(UNSAFE_QUERY_SET_PATH))
    for cwe in CWE_LIST:
        print(f"  {cwe}: {len(unsafe_query_set.get(cwe, []))} 条")
else:
    results_s32 = load_json_stream(str(PROXY_RETRIEVAL_RESULTS_PATH))

    unsafe_query_set = {}
    summary = []
    for cwe in CWE_LIST:
        bfps = collect_unsafe_bfps_for_cwe(
            results_s32, cwe, RELEVANCE_THRESHOLD, TOP_K_FILTER, CVE_ASSOCIATION_THRESHOLD
        )
        unsafe_query_set[cwe] = bfps
        summary.append((cwe, len(bfps)))

    UNSAFE_QUERY_SET_PATH.parent.mkdir(parents=True, exist_ok=True)
    save_json(unsafe_query_set, str(UNSAFE_QUERY_SET_PATH))

    print(f"\n===== 含安全隐患的历史查询集导出汇总 =====")
    print(f"  过滤: score >= {RELEVANCE_THRESHOLD}，rank ∈ [1, {TOP_K_FILTER}]，关联 CVE 数 >= {CVE_ASSOCIATION_THRESHOLD}")
    print(f"  {'-'*30}")
    print(f"  {'CWE':<12} {'unsafe BFP':>12}")
    print(f"  {'-'*30}")
    for cwe, total in summary:
        print(f"  {cwe:<12} {total:>12}")
    print(f"  unsafe → {UNSAFE_QUERY_SET_PATH}")

print("Section 3.2 Done")

2026-07-25 11:41:43 [INFO] src.utils.io: Loaded JSON from <PROJECT_ROOT>/data/query/black/unsafe_query_set_v4.json


[SKIP] 含安全隐患的历史查询集已存在：<PROJECT_ROOT>/data/query/black/unsafe_query_set_v4.json
  CWE-787: 83 条
  CWE-416: 102 条
  CWE-125: 153 条
  CWE-476: 94 条
  CWE-20: 237 条
  CWE-200: 136 条
  CWE-119: 369 条
  CWE-362: 103 条
Section 3.2 Done


## Section 4 — 代理查询集 & 测试集划分

对含安全隐患的历史查询集 `unsafe_query_set_v4.json`，逐 CWE 按 `SPLIT_RATIO`（默认 8:2）在固定随机种子下洗牌切分为代理查询集（前 80%，投毒目标候选池）与测试查询集（后 20%，评估用），分别落盘。

In [10]:
# ========== Section 4: 代理查询集 & 测试集划分 ==========
# 产出：
#   data/query/{BOX}/proxy/proxy_query_set_v4.json  （前 80%，投毒目标候选池）
#   data/query/{BOX}/test/test_query_set_v4.json    （后 20%，评估测试集）

SPLIT_RATIO   = 0.8   # 代理查询集 : 测试集 = 8 : 2
RANDOM_SEED   = 42

PROXY_QUERY_SET_PATH = PROJECT_ROOT / "data/query" / BOX / "proxy" / "proxy_query_set_v4.json"
TEST_QUERY_SET_PATH  = PROJECT_ROOT / "data/query" / BOX / "test" / "test_query_set_v4.json"

if PROXY_QUERY_SET_PATH.exists() and TEST_QUERY_SET_PATH.exists():
    print(f"[SKIP] proxy/test 查询集已存在")
    proxy_query_set = load_json(str(PROXY_QUERY_SET_PATH))
    test_query_set  = load_json(str(TEST_QUERY_SET_PATH))
    for cwe in CWE_LIST:
        print(f"  {cwe}: proxy={len(proxy_query_set.get(cwe, []))}, test={len(test_query_set.get(cwe, []))}")
else:
    # 复用 Section 3.2 的 unsafe_query_set（若内存中不存在则从磁盘加载）
    try:
        unsafe_query_set
    except NameError:
        unsafe_query_set = load_json(str(UNSAFE_QUERY_SET_PATH))

    proxy_query_set = {}
    test_query_set  = {}
    summary_s4 = []

    for cwe in CWE_LIST:
        bfps = unsafe_query_set.get(cwe, [])
        shuffled = bfps[:]
        random.Random(RANDOM_SEED).shuffle(shuffled)
        cut = int(len(shuffled) * SPLIT_RATIO)   # 前 80% 为代理查询集，后 20% 为测试集
        proxy_query_set[cwe] = shuffled[:cut]
        test_query_set[cwe]  = shuffled[cut:]
        summary_s4.append((cwe, len(bfps), len(proxy_query_set[cwe]), len(test_query_set[cwe])))

    PROXY_QUERY_SET_PATH.parent.mkdir(parents=True, exist_ok=True)
    TEST_QUERY_SET_PATH.parent.mkdir(parents=True, exist_ok=True)
    save_json(proxy_query_set, str(PROXY_QUERY_SET_PATH))
    save_json(test_query_set,  str(TEST_QUERY_SET_PATH))

    print(f"\n===== 代理/测试查询集划分汇总 =====")
    print(f"  分割: proxy {SPLIT_RATIO*100:.0f}% / test {100-SPLIT_RATIO*100:.0f}% (seed={RANDOM_SEED})")
    print(f"  {'-'*46}")
    print(f"  {'CWE':<12} {'unsafe':>8} {'proxy':>7} {'test':>7}")
    print(f"  {'-'*46}")
    for cwe, total, p, t in summary_s4:
        print(f"  {cwe:<12} {total:>8} {p:>7} {t:>7}")
    print(f"  proxy → {PROXY_QUERY_SET_PATH}")
    print(f"  test  → {TEST_QUERY_SET_PATH}")

print("Section 4 Done")

2026-07-25 11:41:55 [INFO] src.utils.io: Loaded JSON from <PROJECT_ROOT>/data/query/black/proxy/proxy_query_set_v4.json
2026-07-25 11:41:55 [INFO] src.utils.io: Loaded JSON from <PROJECT_ROOT>/data/query/black/test/test_query_set_v4.json


[SKIP] proxy/test 查询集已存在
  CWE-787: proxy=66, test=17
  CWE-416: proxy=81, test=21
  CWE-125: proxy=122, test=31
  CWE-476: proxy=75, test=19
  CWE-20: proxy=189, test=48
  CWE-200: proxy=108, test=28
  CWE-119: proxy=295, test=74
  CWE-362: proxy=82, test=21
Section 4 Done


## Section 5 — 投毒目标选择与正样本构建

以 `configs/database/bfp/data_prep_v4/target.yml`（APR 受害库）为检索器，对代理查询集逐 CWE 执行：

1. **反向索引**：以每条代理查询的 `buggy_code` 检索 APR 库（`top_k` 由 target.yml 指定，默认 5），统计每个被召回 BFP 关联到的**唯一代理查询集合**；关联数严格 `> τ`（默认 2）的 BFP 纳入候选（`entity` + `retrieval_attack.positive_samples`，正样本即关联到的代理查询）。
2. **预算分配**：`总预算 = round(POISON_RATE × APR_CORPUS_SIZE)`，按各 CWE 候选规模比例分配到各 CWE。
3. **聚类 + 采样**：逐 CWE 对候选 `buggy_code` 用 APR 检索器向量化 + KMeans（yellowbrick 自动选 k，失败回退 `DEFAULT_K`）；按子簇规模分配簇内预算；簇内按关联数加权**无放回**采样。

产出 `data/query/{BOX}/target/poison_targets_set_v4.json`（含 `entity` 与 `retrieval_attack.positive_samples`）。

In [13]:
# ========== Section 5: 投毒目标选择与正样本构建（反向索引 + 聚类 + 预算加权采样） ==========
# 产出：data/query/{BOX}/target/poison_targets_set_v4.json

from yellowbrick.cluster import KElbowVisualizer
import matplotlib
matplotlib.use("Agg")  # 关闭 GUI 后端，避免 notebook 中 yellowbrick 的渲染开销

# TARGET_CWE_LIST       = ["CWE-125", "CWE-20", "CWE-119", "CWE-787", "CWE-416"]
# TARGET_CWE_LIST = ["CWE-787", "CWE-416", "CWE-125", "CWE-476", "CWE-20", "CWE-200"]
# TARGET_CWE_LIST = ["CWE-787", "CWE-416", "CWE-125", "CWE-476", "CWE-20", "CWE-200", "CWE-119", "CWE-190", "CWE-362", "CWE-120"]
TARGET_CWE_LIST = ["CWE-787", "CWE-416", "CWE-125", "CWE-476", "CWE-20", "CWE-200", "CWE-119", "CWE-362"]

ASSOCIATION_THRESHOLD = 2          # 关联数量阈值 τ：保留关联代理查询数 > τ 的 BFP
# POISON_RATE           = 0.008     # 总投毒率 θ, 最终预计投毒0.65
POISON_RATE           = 0.0065     # 总投毒率 θ, 最终预计投毒0.65
APR_CORPUS_SIZE       = 50000      # 与 Section 1.2 的 CORPUS_SIZE 一致
K_RANGE               = (2, 15)    # yellowbrick KElbowVisualizer 搜索范围
DEFAULT_K             = 5          # 无法识别肘点时的回退簇数
RANDOM_SEED_S5        = 42

POISON_TARGETS_OUT = PROJECT_ROOT / "data/query" / BOX / "target" / "poison_targets_set_v4.json"

# 读取 target.yml（APR 受害库；top_k / query_target 等检索参数）
target_db_config = load_yaml(str(PROJECT_ROOT / "configs/database/bfp/data_prep_v4/target.yml"))
if not os.path.isabs(target_db_config["database"]["uri"]):
    target_db_config["database"]["uri"] = str(PROJECT_ROOT / target_db_config["database"]["uri"])

top_k_target        = target_db_config["retrieval"].get("top_k", 5)
query_target_target = target_db_config["retrieval"].get("query_target", "buggy_code")


if POISON_TARGETS_OUT.exists():
    print(f"[SKIP] 投毒目标文件已存在：{POISON_TARGETS_OUT}")
    poison_targets = load_json(str(POISON_TARGETS_OUT))
    for cwe in TARGET_CWE_LIST:
        print(f"  {cwe}: {len(poison_targets.get(cwe, []))} 条")
else:
    # 复用 Section 4 的 proxy_query_set（若内存中不存在则从磁盘加载）
    try:
        proxy_query_set
    except NameError:
        proxy_query_set = load_json(str(PROXY_QUERY_SET_PATH))

    # 打开 APR 库 + 加载 embedder（同时用于反向索引检索与聚类向量化）
    target_milvus    = BFPMilvusClient(target_db_config)
    target_embedder  = load_embedder_from_db_config(target_db_config)
    target_retriever = BFPRetriever(target_embedder, target_milvus)

    # ── 步骤1: 逐 CWE 反向索引 + τ 过滤，收集投毒目标候选 ──
    candidates_by_cwe = {}   # cwe -> [ {entity, positive_samples, assoc_count}, ... ]
    for cwe in TARGET_CWE_LIST:
        proxy_queries = proxy_query_set.get(cwe, [])
        if not proxy_queries:
            candidates_by_cwe[cwe] = []
            print(f"[{cwe}] 无代理查询，跳过")
            continue

        print(f"\n{'='*60}")
        print(f"[{cwe}] 代理查询数: {len(proxy_queries)}，以 {query_target_target} 检索 APR 库 top_k={top_k_target}")

        retrieval_results_t = target_retriever.dense_retrieve(
            proxy_queries, top_k=top_k_target, query_target=query_target_target
        )

        # 反向索引：doc_id -> {proxy_id: proxy_entity}；bfp_hit: doc_id -> 最高分命中
        reverse_index = defaultdict(dict)
        bfp_hit       = {}
        for result in retrieval_results_t:
            proxy_entity = result["entity"]
            for hit in result["retrieval_results"]:
                did = hit["doc_id"]
                reverse_index[did][proxy_entity["id"]] = proxy_entity
                if did not in bfp_hit or hit["score"] > bfp_hit[did]["score"]:
                    bfp_hit[did] = hit

        # τ 过滤：关联到的唯一代理查询数严格 > τ
        cwe_candidates = []
        for did, proxy_map in reverse_index.items():
            assoc_count = len(proxy_map)
            if assoc_count <= ASSOCIATION_THRESHOLD:
                continue
            hit = bfp_hit[did]
            positive_samples = [
                {
                    "id"        : p["id"],
                    "language"  : p.get("language", "C"),
                    "source"    : p.get("source", ""),
                    "buggy_code": p.get("buggy_code", ""),
                    "fixed_code": p.get("fixed_code", ""),
                }
                for p in proxy_map.values()
            ]
            cwe_candidates.append({
                "entity": {
                    "id"        : hit["doc_id"],
                    "language"  : hit.get("language", "C"),
                    "source"    : hit.get("source", ""),
                    "buggy_code": hit.get("buggy_code", ""),
                    "fixed_code": hit.get("fixed_code", ""),
                },
                "positive_samples": positive_samples,
                "assoc_count": assoc_count,
            })
        candidates_by_cwe[cwe] = cwe_candidates
        print(f"[{cwe}] 反向索引候选 BFP: {len(reverse_index)} 个，τ>{ASSOCIATION_THRESHOLD} 过滤后: {len(cwe_candidates)} 个")

    # ── 步骤2: 预算分配（总预算 → 各 CWE 按候选规模比例）──
    total_budget = round(POISON_RATE * APR_CORPUS_SIZE)
    cwe_sizes = {cwe: len(candidates_by_cwe.get(cwe, [])) for cwe in TARGET_CWE_LIST}
    cwe_total = sum(cwe_sizes.values())
    if cwe_total == 0:
        raise ValueError("所有 CWE 候选均为空，无法分配预算")
    cwe_budget = {
        cwe: round(total_budget * cwe_sizes[cwe] / cwe_total)
        for cwe in TARGET_CWE_LIST
    }
    print(f"\n总投毒预算: {total_budget} (POISON_RATE × APR_CORPUS = {POISON_RATE}×{APR_CORPUS_SIZE})")
    print(f"各 CWE 候选规模: {cwe_sizes}")
    print(f"各 CWE 投毒预算: {cwe_budget}")

    # ── 步骤3: 逐 CWE 聚类 + 簇内预算分配 + 关联数加权无放回采样 ──
    rng_s5 = np.random.default_rng(RANDOM_SEED_S5)
    poison_targets = {}

    for cwe in TARGET_CWE_LIST:
        subset = candidates_by_cwe.get(cwe, [])
        budget = cwe_budget[cwe]
        if not subset or budget <= 0:
            print(f"\n[{cwe}] 候选为空或预算为零，跳过")
            poison_targets[cwe] = []
            continue

        print(f"\n{'='*60}")
        print(f"[{cwe}] 候选规模 {len(subset)}，预算 {budget}")

        # 以候选 buggy_code 向量化（使用 APR 检索器模型）
        codes_cwe = [c["entity"]["buggy_code"] for c in subset]
        emb_cwe   = np.asarray(target_embedder.embed_documents(codes_cwe))

        # yellowbrick 自动选 k
        k_max_eff = max(2, min(K_RANGE[1], len(subset) - 1))
        try:
            visualizer = KElbowVisualizer(
                KMeans(n_init="auto", random_state=RANDOM_SEED_S5),
                k=(K_RANGE[0], k_max_eff + 1),
                timings=False,
                force_model=True,
            )
            visualizer.fit(emb_cwe)
            best_k = int(visualizer.elbow_value_) if visualizer.elbow_value_ else DEFAULT_K
        except Exception as e:
            print(f"  yellowbrick 选 k 失败 ({e})，回退 DEFAULT_K={DEFAULT_K}")
            best_k = DEFAULT_K
        best_k = max(1, min(best_k, len(subset)))
        print(f"  自动选定 k = {best_k}")

        # KMeans 聚类
        kmeans_cwe = KMeans(n_clusters=best_k, n_init=10, random_state=RANDOM_SEED_S5)
        cluster_labels = kmeans_cwe.fit_predict(emb_cwe)

        clusters = defaultdict(list)
        for idx, label in enumerate(cluster_labels):
            clusters[int(label)].append(subset[idx])

        # 簇内预算分配（按簇规模）+ 簇内按关联数加权无放回采样
        cluster_sizes  = {c: len(items) for c, items in clusters.items()}
        cwe_total_size = sum(cluster_sizes.values())
        targets_cwe = []
        used_total  = 0
        for cluster_id, items in clusters.items():
            cluster_quota = round(budget * cluster_sizes[cluster_id] / cwe_total_size)
            cluster_quota = min(cluster_quota, len(items))
            if cluster_quota <= 0:
                continue
            # 关联数（正样本数）越多，投毒目标被抽中的概率越大
            weights = np.array([max(it["assoc_count"], 1) for it in items], dtype=np.float64)
            probs   = weights / weights.sum()
            idx_sel = rng_s5.choice(len(items), size=cluster_quota, replace=False, p=probs)
            for i in idx_sel:
                it = items[int(i)]
                targets_cwe.append({
                    "entity": it["entity"],
                    "retrieval_attack": {"positive_samples": it["positive_samples"]},
                })
            used_total += cluster_quota
            print(f"  簇 {cluster_id}: 大小 {cluster_sizes[cluster_id]:>4d}，配额 {cluster_quota:>3d}")

        poison_targets[cwe] = targets_cwe
        print(f"  [{cwe}] 总投毒目标: {len(targets_cwe)} 条（预算 {budget}，实际命中 {used_total}）")

    # 关闭 Milvus + 释放 embedder
    target_milvus.close()
    release_embedder(target_embedder)

    # ── 步骤4: 写出 ──
    POISON_TARGETS_OUT.parent.mkdir(parents=True, exist_ok=True)
    save_json(poison_targets, str(POISON_TARGETS_OUT))
    print(f"\n✅ 投毒目标文件已保存: {POISON_TARGETS_OUT}")

    print(f"\n===== 汇总 =====")
    print(f"  {'CWE':<12} {'候选数':>8} {'预算':>6} {'命中':>6}")
    print(f"  {'-'*36}")
    for cwe in TARGET_CWE_LIST:
        print(f"  {cwe:<12} {cwe_sizes[cwe]:>8} {cwe_budget[cwe]:>6} {len(poison_targets.get(cwe, [])):>6}")

print("\nSection 5 Done — 投毒目标选择完成 ✅")

2026-07-25 12:00:31 [INFO] src.rag.bfp.milvus_client: 从模型配置 <PROJECT_ROOT>/configs/models/harrier-oss-v1-0.6b.yml 读取 dim=1024
2026-07-25 12:00:31 [INFO] src.rag.bfp.milvus_client: Connecting to Milvus at <PROJECT_ROOT>/data/milvus/bfp/apr/harrier-oss-v1-0.6b.db
2026-07-25 12:00:31 [INFO] src.rag.bfp.milvus_client: Milvus client connected
2026-07-25 12:00:31 [INFO] src.utils.device: nvidia-smi 探测完成: 健康 GPU = [0]
2026-07-25 12:00:31 [WARNING] src.utils.device: 设备池预筛选排除 cuda:1（nvidia-smi 判定为不健康）
2026-07-25 12:00:31 [WARNING] src.utils.device: 设备池预筛选排除 cuda:2（nvidia-smi 判定为不健康）
2026-07-25 12:00:31 [WARNING] src.utils.device: 设备池预筛选排除 cuda:3（nvidia-smi 判定为不健康）
2026-07-25 12:00:31 [WARNING] src.utils.device: 设备池预筛选排除 cuda:4（nvidia-smi 判定为不健康）
2026-07-25 12:00:31 [WARNING] src.utils.device: 设备池预筛选排除 cuda:5（nvidia-smi 判定为不健康）
2026-07-25 12:00:31 [WARNING] src.utils.device: 设备池预筛选排除 cuda:6（nvidia-smi 判定为不健康）
2026-07-25 12:00:31 [WARNING] src.utils.device: 设备池预筛选排除 cuda:7（nvidia-smi 判定为不健康）
2026


[CWE-787] 代理查询数: 66，以 buggy_code 检索 APR 库 top_k=10


APR dense batch retrieval: 100%|██████████| 1/1 [00:08<00:00,  8.75s/it]
2026-07-25 12:00:50 [INFO] src.rag.bfp.retriever: Dense batch retrieval completed: 66 queries, top_k=10, shards=1


[CWE-787] 反向索引候选 BFP: 307 个，τ>2 过滤后: 82 个

[CWE-416] 代理查询数: 81，以 buggy_code 检索 APR 库 top_k=10


APR dense batch retrieval: 100%|██████████| 1/1 [00:04<00:00,  4.16s/it]
2026-07-25 12:00:54 [INFO] src.rag.bfp.retriever: Dense batch retrieval completed: 81 queries, top_k=10, shards=1


...

  自动选定 k = 10
  簇 7: 大小   54，配额  15
  簇 2: 大小   31，配额   8
  簇 0: 大小   44，配额  12
  簇 4: 大小   22，配额   6
  簇 6: 大小   52，配额  14
  簇 8: 大小   40，配额  11
  簇 1: 大小   50，配额  14
  簇 9: 大小   20，配额   5
  簇 3: 大小   10，配额   3
  簇 5: 大小   25，配额   7
  [CWE-119] 总投毒目标: 95 条（预算 94，实际命中 95）

[CWE-362] 候选规模 88，预算 24


Batches:  44%|████▍     | 39/88 [00:03<00:02, 21.95it/s]I0000 00:00:1784952281.161160   12377 chttp2_transport.cc:1353] unix:/tmp/tmpl3vqwqjg_harrier-oss-v1-0.6b.db.sock: Got goaway [11] err=UNAVAILABLE:GOAWAY received; Error code: 11; Debug Text: too_many_pings {http2_error:11, grpc_status:14}
E0000 00:00:1784952281.161263   12377 chttp2_transport.cc:1385] unix:/tmp/tmpl3vqwqjg_harrier-oss-v1-0.6b.db.sock: Received a GOAWAY with error code ENHANCE_YOUR_CALM and debug data equal to "too_many_pings". Current keepalive time (before throttling): 10000ms
Batches: 100%|██████████| 88/88 [00:05<00:00, 17.15it/s]
2026-07-25 12:04:42 [INFO] src.rag.bfp.milvus_client: Milvus client closed


  自动选定 k = 7
  簇 2: 大小   29，配额   8
  簇 5: 大小   12，配额   3
  簇 0: 大小    7，配额   2
  簇 6: 大小    8，配额   2
  簇 4: 大小   13，配额   4
  簇 1: 大小   13，配额   4
  簇 3: 大小    6，配额   2
  [CWE-362] 总投毒目标: 25 条（预算 24，实际命中 25）
[GPU] 嵌入模型已释放


2026-07-25 12:04:43 [INFO] src.utils.io: Saved results to <PROJECT_ROOT>/data/query/black/target/poison_targets_set_v4.json



✅ 投毒目标文件已保存: <PROJECT_ROOT>/data/query/black/target/poison_targets_set_v4.json

===== 汇总 =====
  CWE               候选数     预算     命中
  ------------------------------------
  CWE-787            82     22     21
  CWE-416            93     25     25
  CWE-125           143     39     39
  CWE-476            71     19     19
  CWE-20            253     69     69
  CWE-200           122     33     32
  CWE-119           348     94     95
  CWE-362            88     24     25

Section 5 Done — 投毒目标选择完成 ✅


## Section 6 — 相关漏洞检索

对每个投毒目标，以 `entity.fixed_code` 在 Vul-Vinj 库（`vinj.yml`）中检索目标 CWE 的相关 CVE 实例（Top-`TOP_K_VINJ`，`cwe_list` 过滤），再以**不放回加权抽样**（初始权重为相关性分数 `score`，每抽中一个 `cve_id` 后将剩余同 `cve_id` 候选权重乘以衰减因子 `RELEVANT_VUL_DECAY_GAMMA` 以提升多样性）选出 `RELEVANT_VUL_NUM` 条，回写为 `generation_attack.vinj_attack.relevant_vul`（`vul_pattern` 置空，待后续漏洞注入阶段填充），并置空 `jailbreak_attack`。

> `retrieval_attack.positive_samples` 沿用 Section 5 的结果，本节不重算。

In [16]:
# ========== Section 6: 相关漏洞检索（Vinj 检索 + 加权采样 relevant_vul） ==========

RELEVANT_VUL_NUM         = 5
TOP_K_VINJ               = 15
RANDOM_SEED_S6           = 42
RELEVANT_VUL_DECAY_GAMMA = 0.9   # 同 cve_id 多样性衰减因子 γ ∈ (0,1)


def weighted_sample_relevant_vul(hits, num, gamma, rng):
    """不放回加权抽样 relevant_vul（序贯 + 同 cve_id 多样性衰减）。

    初始权重 = 相关性分数（score）；每抽中一个候选后，将剩余“同 cve_id”
    候选的权重乘以衰减因子 gamma，从而实现“抽到相同 cve_id 越多，
    同属该 cve_id 但未被抽取的候选权重越低”的多样性约束。
    """
    pool     = list(hits)
    weights  = [max(float(h.get("score", 0.0)), 1e-6) for h in pool]
    cves     = [(h.get("cve_id") or "").strip() for h in pool]
    selected = []
    while pool and len(selected) < num:
        total = sum(weights)
        if total <= 0:
            break
        probs = np.array(weights, dtype=np.float64) / total
        idx = int(np.random.default_rng(rng.randint(0, 2**31 - 1)).choice(len(pool), p=probs))
        chosen_cve = cves[idx]
        selected.append(pool.pop(idx))
        weights.pop(idx)
        cves.pop(idx)
        for j, c in enumerate(cves):       # 同 cve_id 衰减
            if c and c == chosen_cve:
                weights[j] *= gamma
    return selected


def _is_relevant_vul_enriched(targets_dict) -> bool:
    """字段级幂等检查：首个含 target 的 CWE 首条记录是否已含 relevant_vul。"""
    for cwe, items in targets_dict.items():
        if not items:
            continue
        head = items[0]
        return bool(head.get("generation_attack", {}).get("vinj_attack", {}).get("relevant_vul"))
    return False


poison_targets = load_json(str(POISON_TARGETS_OUT))

if _is_relevant_vul_enriched(poison_targets):
    print(f"[SKIP] 投毒目标文件已含 relevant_vul 字段：{POISON_TARGETS_OUT}")
else:
    # 打开 Vul-Vinj 库 + embedder
    vinj_db_config_s6 = load_yaml(str(PROJECT_ROOT / "configs/database/vul/vinj.yml"))
    if not os.path.isabs(vinj_db_config_s6["database"]["uri"]):
        vinj_db_config_s6["database"]["uri"] = str(PROJECT_ROOT / vinj_db_config_s6["database"]["uri"])

    vinj_milvus_s6    = VulMilvusClient(vinj_db_config_s6)
    embedder_s6       = load_embedder_from_db_config(vinj_db_config_s6)
    vinj_retriever_s6 = VulRetriever(embedder_s6, vinj_milvus_s6)

    rng_s6 = random.Random(RANDOM_SEED_S6)
    summary_s6 = []

    for cwe in TARGET_CWE_LIST:
        targets = poison_targets.get(cwe, [])
        if not targets:
            summary_s6.append((cwe, 0, 0))
            continue

        print(f"\n{'='*60}")
        print(f"[{cwe}] 投毒目标数: {len(targets)}")

        # 批量检索：以 entity.fixed_code 为查询，CWE 过滤
        queries_s6 = [
            {
                "id"        : t["entity"]["id"],
                "language"  : t["entity"].get("language", "C"),
                "source"    : t["entity"].get("source", ""),
                "buggy_code": t["entity"].get("buggy_code", ""),
                "fixed_code": t["entity"]["fixed_code"],
            }
            for t in targets
        ]
        retrieved = vinj_retriever_s6.dense_retrieve(
            queries_s6,
            top_k=TOP_K_VINJ,
            query_target="fixed_code",
            cwe_list=[cwe],
        )
        ret_map = {r["id"]: r.get("retrieval_results", []) for r in retrieved}

        n_with_vul = 0
        for tgt in targets:
            ent_id = tgt["entity"]["id"]
            hits = ret_map.get(ent_id, [])
            # dense_retrieve 已用 cwe 过滤，此处再做一次防御性检查
            filtered_hits = [h for h in hits if h.get("cwe_id", "") == cwe]
            cwe_hits = weighted_sample_relevant_vul(
                filtered_hits, RELEVANT_VUL_NUM, RELEVANT_VUL_DECAY_GAMMA, rng_s6
            )

            relevant_vul = []
            for h in cwe_hits:
                relevant_vul.append({
                    "doc_id"      : h.get("doc_id"),
                    "chunk_id"    : h.get("chunk_id", 0),
                    "cluster_id"  : h.get("cluster_id", ""),
                    "language"    : h.get("language", ""),
                    "source"      : h.get("source", ""),
                    "total_chunks": h.get("total_chunks", 1),
                    "vul_code"    : h.get("vul_code", ""),
                    "cwe_id"      : h.get("cwe_id", ""),
                    "cve_id"      : h.get("cve_id", ""),
                    "cwe_desc"    : h.get("cwe_desc", ""),
                    "cve_desc"    : h.get("cve_desc", ""),
                    "vul_pattern" : "",
                    "score"       : h.get("score", 0.0),
                })

            if relevant_vul:
                n_with_vul += 1

            # 保留 Section 5 的 positive_samples，仅补充 generation_attack
            tgt.setdefault("retrieval_attack", {"positive_samples": []})
            tgt["generation_attack"] = {
                "vinj_attack"     : {"relevant_vul": relevant_vul},
                "jailbreak_attack": {},
            }

        summary_s6.append((cwe, len(targets), n_with_vul))
        print(f"  含 relevant_vul: {n_with_vul}/{len(targets)}")

    vinj_milvus_s6.close()
    release_embedder(embedder_s6)

    save_json(poison_targets, str(POISON_TARGETS_OUT))
    print(f"\n✅ 投毒目标文件（含 relevant_vul）已回写: {POISON_TARGETS_OUT}")

    print(f"\n===== 汇总 =====")
    print(f"  {'CWE':<12} {'目标数':>6} {'含 vul':>8}")
    print(f"  {'-'*30}")
    for cwe, n_tot, n_vul in summary_s6:
        print(f"  {cwe:<12} {n_tot:>6} {n_vul:>8}")

print("\nSection 6 Done — 相关漏洞检索完成，数据准备管线全部完成 ✅")

2026-07-25 12:07:52 [INFO] src.utils.io: Loaded JSON from <PROJECT_ROOT>/data/query/black/target/poison_targets_set_v4.json
2026-07-25 12:07:52 [INFO] src.rag.vul.milvus_client: 从模型配置 <PROJECT_ROOT>/configs/models/harrier-oss-v1-0.6b.yml 读取 dim=1024
2026-07-25 12:07:52 [INFO] src.rag.vul.milvus_client: Connecting to Milvus at <PROJECT_ROOT>/data/milvus/vul/harrier-oss-v1-0.6b.db
2026-07-25 12:07:52 [INFO] src.rag.vul.milvus_client: Milvus client connected
2026-07-25 12:07:52 [INFO] src.utils.device: nvidia-smi 探测完成: 健康 GPU = [0]
2026-07-25 12:07:52 [WARNING] src.utils.device: 设备池预筛选排除 cuda:1（nvidia-smi 判定为不健康）
2026-07-25 12:07:52 [WARNING] src.utils.device: 设备池预筛选排除 cuda:2（nvidia-smi 判定为不健康）
2026-07-25 12:07:52 [WARNING] src.utils.device: 设备池预筛选排除 cuda:3（nvidia-smi 判定为不健康）
2026-07-25 12:07:52 [WARNING] src.utils.device: 设备池预筛选排除 cuda:4（nvidia-smi 判定为不健康）
2026-07-25 12:07:52 [WARNING] src.utils.device: 设备池预筛选排除 cuda:5（nvidia-smi 判定为不健康）
2026-07-25 12:07:52 [WARNING] src.utils.device: 设备


[CWE-787] 投毒目标数: 21


Vul dense batch retrieval:   0%|          | 0/1 [00:00<?, ?it/s]I0000 00:00:1784952508.468963    6999 chttp2_transport.cc:1353] unix:/tmp/tmp40hyyc5s_harrier-oss-v1-0.6b.db.sock: Got goaway [11] err=UNAVAILABLE:GOAWAY received; Error code: 11; Debug Text: too_many_pings {http2_error:11, grpc_status:14}
E0000 00:00:1784952508.469497    6999 chttp2_transport.cc:1385] unix:/tmp/tmp40hyyc5s_harrier-oss-v1-0.6b.db.sock: Received a GOAWAY with error code ENHANCE_YOUR_CALM and debug data equal to "too_many_pings". Current keepalive time (before throttling): 10000ms
Vul dense batch retrieval: 100%|██████████| 1/1 [00:23<00:00, 23.23s/it]
2026-07-25 12:08:51 [INFO] src.rag.vul.retriever: Dense batch retrieval completed: 21 queries, top_k=15, cluster_search=False, shards=1


  含 relevant_vul: 21/21

[CWE-416] 投毒目标数: 25


Vul dense batch retrieval: 100%|██████████| 1/1 [00:00<00:00, 24.75it/s]
2026-07-25 12:08:53 [INFO] src.rag.vul.retriever: Dense batch retrieval completed: 25 queries, top_k=15, cluster_search=False, shards=1


...

  含 relevant_vul: 95/95

[CWE-362] 投毒目标数: 25


Vul dense batch retrieval: 100%|██████████| 1/1 [00:00<00:00, 31.52it/s]
2026-07-25 12:09:24 [INFO] src.rag.vul.retriever: Dense batch retrieval completed: 25 queries, top_k=15, cluster_search=False, shards=1
2026-07-25 12:09:24 [INFO] src.rag.vul.milvus_client: Milvus client closed


  含 relevant_vul: 25/25
[GPU] 嵌入模型已释放


2026-07-25 12:09:26 [INFO] src.utils.io: Saved results to <PROJECT_ROOT>/data/query/black/target/poison_targets_set_v4.json



✅ 投毒目标文件（含 relevant_vul）已回写: <PROJECT_ROOT>/data/query/black/target/poison_targets_set_v4.json

===== 汇总 =====
  CWE             目标数    含 vul
  ------------------------------
  CWE-787          21       21
  CWE-416          25       25
  CWE-125          39       39
  CWE-476          19       19
  CWE-20           69       69
  CWE-200          32       32
  CWE-119          95       95
  CWE-362          25       25

Section 6 Done — 相关漏洞检索完成，数据准备管线全部完成 ✅


## Appendix — 投毒目标正样本关联数分布统计

上下文独立单元格：读取任意投毒目标 JSON（如 `data/query/black/target/gte/poison_targets_set_v4_tau_2_top_5.json`），按 CWE 统计各投毒目标关联到的代理查询（`retrieval_attack.positive_samples`）数量分布。

In [17]:
# ========== Appendix: 投毒目标正样本关联数分布统计（上下文独立） ==========
# 修改 POISON_TARGETS_FILE 后直接运行本单元格即可，无需先跑前面的 Section。

import os
import sys
import json
from pathlib import Path
from collections import Counter

# ── 输入：投毒目标文件（相对项目根或绝对路径均可）──
POISON_TARGETS_FILE = "data/query/black/target/poison_targets_set_v4.json"

# ── 项目根目录上溯（本单元格自包含，不依赖 Section 0）──
_ROOT = Path(os.path.abspath("")).resolve()
while not (_ROOT / "configs").exists() and _ROOT != _ROOT.parent:
    _ROOT = _ROOT.parent
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

_path = Path(POISON_TARGETS_FILE)
if not _path.is_absolute():
    _path = _ROOT / _path
if not _path.exists():
    raise FileNotFoundError(f"投毒目标文件不存在: {_path}")

with open(_path, encoding="utf-8") as f:
    poison_data = json.load(f)

if not isinstance(poison_data, dict):
    raise ValueError(f"期望顶层为 {{CWE: [targets...]}} 字典，实际类型: {type(poison_data)}")

print(f"投毒目标文件: {_path}")
print(f"CWE 分区数: {len(poison_data)}")
print()

def _assoc_count(tgt: dict) -> int:
    pos = (tgt.get("retrieval_attack") or {}).get("positive_samples") or []
    return len(pos)


def _print_assoc_dist(label: str, targets: list) -> None:
    """打印某一分区（单 CWE 或 CWE-MIXED）的正样本关联数分布。"""
    n = len(targets)
    print(f"{'=' * 60}")
    print(f"[{label}] 投毒目标总数: {n}")
    if n == 0:
        print("  （空）")
        return

    assoc_vals = [_assoc_count(tgt) for tgt in targets]
    assoc_counter = Counter(assoc_vals)

    print(f"  {'关联数 k':>10}  {'目标数':>8}  {'占比':>10}")
    print(f"  {'-' * 34}")
    for k in sorted(assoc_counter.keys()):
        cnt = assoc_counter[k]
        pct = 100.0 * cnt / n
        print(f"  {k:>10}  {cnt:>8}  {pct:>9.2f}%")

    mean_k = sum(assoc_vals) / n
    print(f"  摘要: min={min(assoc_vals)}, max={max(assoc_vals)}, mean={mean_k:.2f}")


# ── 逐 CWE 统计 ──
all_targets = []
for cwe in sorted(poison_data.keys()):
    targets = poison_data[cwe] or []
    all_targets.extend(targets)
    _print_assoc_dist(cwe, targets)

# ── CWE-MIXED：整个投毒目标文件的全局汇总 ──
_print_assoc_dist("CWE-MIXED", all_targets)

print(f"\n{'=' * 60}")
print(f"Done ✅")


投毒目标文件: <PROJECT_ROOT>/data/query/black/target/poison_targets_set_v4.json
CWE 分区数: 8

[CWE-119] 投毒目标总数: 95
       关联数 k       目标数          占比
  ----------------------------------
           3        21      22.11%
           4        17      17.89%
           5        19      20.00%
           6        13      13.68%
           7         8       8.42%
           8         1       1.05%
           9         5       5.26%
          10         6       6.32%
          12         1       1.05%
          13         1       1.05%
          17         1       1.05%
          18         2       2.11%
  摘要: min=3, max=18, mean=5.80
[CWE-125] 投毒目标总数: 39
       关联数 k       目标数          占比
  ----------------------------------
           3        11      28.21%
           4         6      15.38%
           5         9      23.08%
           6         4      10.26%
           7         3       7.69%
           8         3       7.69%
           9         1       2.56%
          10         2       5.1